In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split

In [3]:
ruta = "/content/drive/My Drive/Maestria/Mineria de datos/Dataset tareas/base-de-datos-violencia-intrafamiliar-ano-2024_v3.xlsx"

In [4]:
data = pd.read_excel(ruta)

In [5]:
data.head()

,HEC_DIA,HEC_MES,HEC_ANO,HEC_DEPTO,HEC_DEPTOMCPIO,HEC_TIPAGRE,NUMERO_BOLETA,DIA_EMISION,MES_EMISION,ANO_EMISION,...,ARTICULOCODPEN2,ARTICULOCODPEN3,ARTICULOCODPEN4,ARTICULOTRAS1,ARTICULOTRAS2,ARTICULOTRAS3,ARTICULOTRAS4,MEDIDAS_SEGURIDAD,TIPO_MEDIDA,ORGANISMO_REMITE
0,4,11,2024,1,110,1122,367,4,11,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,24,3,2024,2,202,1222,5,25,3,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,IJ,18.0
2,99,99,9999,1,101,1122,430,2,3,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,28,3,2024,2,202,1122,6,28,3,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,AIJ,18.0
4,12,7,2024,7,706,2122,16,24,7,2024,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,IJ,18.0


In [6]:
data["riesgo_alto"] = data["HEC_DEPTOMCPIO"].apply(lambda x: 1 if x == 101 else 0)


In [7]:
X = data[[
    "VIC_TRABAJA",
    "VIC_ESCOLARIDAD",
    "VIC_EST_CIV",
    "AGR_ESCOLARIDAD",
    "AGR_TRABAJA",
    "AGR_EST_CIV"
]].copy()

X = X.fillna(0)

y = data["riesgo_alto"].values

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

model = Sequential()
model.add(Dense(8, input_dim=6, activation="relu"))
model.add(Dense(4, activation="relu"))
model.add(Dense(1, activation="sigmoid"))

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
history = model.fit(
    X_train,
    y_train,
    epochs=80,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/80
458/458 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.1953 - loss: 8.7547 - val_accuracy: 0.8623 - val_loss: 0.6097
Epoch 2/80
458/458 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8739 - loss: 0.5796 - val_accuracy: 0.8966 - val_loss: 0.5007
Epoch 3/80
458/458 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8962 - loss: 0.4838 - val_accuracy: 0.8998 - val_loss: 0.4372
Epoch 4/80
458/458 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8996 - loss: 0.4260 - val_accuracy: 0.9014 - val_loss: 0.3934
Epoch 5/80
458/458 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9001 - loss: 0.3870 - val_accuracy: 0.9015 - val_loss: 0.3643
Epoch 6/80
458/458 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9025 - loss: 0.3578 - val_accuracy: 0.9015 - val_loss: 0.3458
Epoch 7/80
458/458 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9023 - loss: 0.3413 - val_accuracy: 0.9015 - val_loss: 0.3345
Epoch 8/80
458/458 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8988 - loss: 0.3376 - val_accuracy: 0.

In [10]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Loss en test: {loss:.4f}")
print(f"Accuracy en test: {accuracy:.4f}")

Loss en test: 0.3217
Accuracy en test: 0.9015


In [11]:
hecho = np.array([[1, 21, 2, 21, 1, 2]])

prob_riesgo = model.predict(hecho)[0][0]
clase_predicha = 1 if prob_riesgo >= 0.5 else 0

print("\n=== PREDICCIÓN PARA EL CASO EJEMPLO ===")
print(f"Probabilidad de riesgo_alto (municipio 101): {prob_riesgo:.4f}")
print(f"Clase predicha (umbral 0.5): {clase_predicha}")

if clase_predicha == 1:
    print("Interpretación: el modelo estima ALTO riesgo de que el caso ocurra en un municipio de alta incidencia (101).")
else:
    print("Interpretación: el modelo estima BAJO riesgo; lo más probable es que el caso NO sea del municipio 101.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step

=== PREDICCIÓN PARA EL CASO EJEMPLO ===
Probabilidad de riesgo_alto (municipio 101): 0.0984
Clase predicha (umbral 0.5): 0
Interpretación: el modelo estima BAJO riesgo; lo más probable es que el caso NO sea del municipio 101.
